# Gemma-WorkOrder：一键完整实验\n\n在 Colab 顶部选择 **运行时 → 更改运行时类型 → T4 GPU**，然后依次运行全部单元格（菜单：运行时 → 全部运行）。\n\n本 Notebook 使用 90 条受控自建样本，验证 Gemma 3-1B 在工单 JSON 抽取与本地只读工具路由任务上的 Base-vs-QLoRA 差异；它不用于工业故障诊断、自动派单或维修决策。

In [ ]:
# 读取 Hugging Face Token，并检查 GPU。\n# 先在 Colab 左侧 Secrets（钥匙图标）创建 HF_TOKEN 并打开 Notebook access。\nimport os\nfrom google.colab import userdata\n\ntoken = userdata.get('HF_TOKEN')\nif not token:\n    raise RuntimeError('未读取到 HF_TOKEN：请在 Colab Secrets 中创建它并开启 Notebook access。')\nos.environ['HF_TOKEN'] = token\n!nvidia-smi

In [ ]:
# 下载/更新项目，并安装依赖。\n# 卸载 Colab 预装的旧 torchao：它会与 PEFT 的 Adapter 合并冲突。\nfrom pathlib import Path\n\nrepo = Path('/content/gemma-inference-eval')\nif not repo.exists():\n    !git clone -b agent/gemma-workorder-roadmap https://github.com/zmh2245749337/gemma-inference-eval.git /content/gemma-inference-eval\nelse:\n    !git -C /content/gemma-inference-eval fetch origin agent/gemma-workorder-roadmap\n    !git -C /content/gemma-inference-eval checkout agent/gemma-workorder-roadmap\n    !git -C /content/gemma-inference-eval pull origin agent/gemma-workorder-roadmap\n%cd /content/gemma-inference-eval\n!pip uninstall -y torchao || true\n!pip -q install -r requirements-colab.txt

In [ ]:
# 构建固定随机种子的 90 条受控自建数据。\n!python scripts/build_workorder_dataset.py --output-dir data/workorder --samples 90 --seed 42

In [ ]:
# 对照组：Base Gemma（未经本任务微调）。\n!python scripts/evaluate_workorder.py --model-id google/gemma-3-1b-it --precision 4bit --dataset data/workorder/test.jsonl --output reports/workorder_base_4bit.json

In [ ]:
# QLoRA：只训练约 1.29% 的可训练参数。\n!python scripts/train_workorder_qlora.py --model-id google/gemma-3-1b-it --train data/workorder/train.jsonl --validation data/workorder/validation.jsonl --output-dir artifacts/workorder_qlora_adapter --epochs 3 --learning-rate 2e-4 --batch-size 1 --gradient-accumulation 8 --max-length 1024 --seed 42

In [ ]:
# 实验组：加载 QLoRA Adapter 后，评测相同的 14 条测试样本。\n!python scripts/evaluate_workorder.py --model-id google/gemma-3-1b-it --adapter artifacts/workorder_qlora_adapter --precision 4bit --dataset data/workorder/test.jsonl --output reports/workorder_qlora_4bit.json

In [ ]:
# 合并 Adapter；此步会自动避开旧 torchao 与 PEFT 的兼容问题。\n!python scripts/merge_workorder_adapter.py --model-id google/gemma-3-1b-it --adapter artifacts/workorder_qlora_adapter --output-dir artifacts/workorder_merged

In [ ]:
# 将合并后的 checkpoint 接入自研 Gemma Decoder，做层级/Logits 白盒数值对齐。\n!python scripts/run_core_alignment.py --model-id artifacts/workorder_merged --precision fp16 --output reports/workorder_merged_core_alignment.json

In [ ]:
# 汇总 Base-vs-QLoRA 指标。只将这里实际跑出的数字写进简历或复盘。\nimport json\nfrom pathlib import Path\n\nfor label, filename in [('Base Gemma', 'reports/workorder_base_4bit.json'), ('QLoRA Gemma', 'reports/workorder_qlora_4bit.json')]:\n    report = json.loads(Path(filename).read_text(encoding='utf-8'))\n    print(f'\n{label}')\n    for key, value in report['metrics'].items():\n        print(f'  {key}: {value}')\n\nalignment = json.loads(Path('reports/workorder_merged_core_alignment.json').read_text(encoding='utf-8'))\nprint('\n白盒对齐最后 Token top-1 一致：', alignment['last_token_logits']['top1_match'])

## 完成后如何表述\n\n可以如实说：在 90 条固定随机种子的受控自建样本上，以 Base Gemma 为对照，使用 4bit QLoRA 微调 Gemma 3-1B 的工单 JSON 抽取任务；同时用 JSON Schema、工具白名单和人工确认草稿约束输出，并将合并后的 checkpoint 接回白盒 Decoder 做数值对齐。\n\n不要说：企业生产数据、工业故障诊断、自动维修或泛化能力已被证明。